In [1]:
print('hallow world')

hallow world


In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta, time
import os
import os
import sys
import pandas as pd
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")
import plotly.express as px

In [17]:
dataset_path = r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw\x_data.xlsx'

In [18]:
main_df = pd.read_excel(r'C:\Users\TPWODL\New folder_Content\AutonomousDataAnalystAgent\data\raw\x_data.xlsx')

In [83]:
import pandas as pd
from datetime import datetime, time

def close_power_outage_duration(dataset_path, selected_day):
    # Read dataset
    main_df = pd.read_excel(dataset_path)
    df = main_df.copy()
    
    # Ensure DATE column is datetime
    df['DATE'] = pd.to_datetime(df['DATE'])
    
    # Filter for specific day
    day_filter_data = df[df['DATE'].dt.date == pd.to_datetime(selected_day).date()].copy()
    
    # Normalize complaint type text
    day_filter_data['COMPLAINT TYPE'] = (
        day_filter_data['COMPLAINT TYPE'].astype(str).str.strip().str.title()
    )

    # Filter complaint types
    power_outage_data = day_filter_data[
        day_filter_data['COMPLAINT TYPE'].isin(['Power Outage', 'No Power Supply'])
    ].copy()

    
    # Convert time objects into datetime for subtraction
    def to_datetime(t):
        if pd.isnull(t):
            return pd.NaT
        if isinstance(t, datetime):
            return t
        if isinstance(t, time):
            return datetime.combine(datetime.today(), t)
        return pd.to_datetime(t)
    
    # Apply conversion first
    power_outage_data.loc[:, 'COMPLAINT_RECEIVED_DT'] = power_outage_data['COMPLAINT RECEIVED TIME'].apply(to_datetime)
    power_outage_data.loc[:, 'FINAL_RESPONSE_DT'] = power_outage_data['FINAL RESPONSE TIME'].apply(to_datetime)
    
    # Fill missing FINAL RESPONSE TIME with current timestamp AFTER conversion
    power_outage_data.loc[:, 'FINAL_RESPONSE_DT'] = power_outage_data['FINAL_RESPONSE_DT'].fillna(pd.Timestamp.now())
    
    # Calculate difference in hours
    power_outage_data.loc[:, 'DURATION_HOURS'] = (
        power_outage_data['FINAL_RESPONSE_DT'] - power_outage_data['COMPLAINT_RECEIVED_DT']
    ).dt.total_seconds() / 3600
    
    # Round to nearest whole hour
    power_outage_data.loc[:, 'DURATION_HOURS_ROUNDED'] = power_outage_data['DURATION_HOURS'].round()
    
    # Integer hours (floor)
    power_outage_data.loc[:, 'DURATION_HOURS_INT'] = power_outage_data['DURATION_HOURS'].fillna(0).astype(int)
    
    # Select relevant columns
    close_open_hour = power_outage_data[['DIVISION', 'SUB-DIVISION', 'SHIFT DUTY', 'CLOSED/OPEN', 'DURATION_HOURS_INT']].copy()
    
    # Define classification function - FIXED logic
    def classify_duration(x):
        if x < 2:  # Fixed: 0-1 hours go here
            return "<2"
        elif 2 <= x < 4:  # Fixed: 2-3 hours go here
            return "2<4"
        elif 4 <= x < 8:  # Fixed: 4-7 hours go here
            return "4<8"
        else:  # x >= 8
            return ">8"

    # Apply classification
    close_open_hour.loc[:, "DURATION_RANGE"] = close_open_hour["DURATION_HOURS_INT"].apply(classify_duration)

    # Pivot table
    pivot = pd.pivot_table(
        close_open_hour,
        values='DURATION_HOURS_INT',
        index=['DIVISION', 'SUB-DIVISION', 'SHIFT DUTY'],
        columns=['DURATION_RANGE', 'CLOSED/OPEN'],
        aggfunc='count',
        fill_value=0,
        margins=True,          
        observed=False
    )
    
    pivot_df = pivot.reset_index()
    pivot_df.columns = ['_'.join([str(c) for c in col if c]) for col in pivot_df.columns.values]

    pivot_df_siftA = pivot_df[pivot_df['SHIFT DUTY'] == 'A']
    pivot_df_siftB = pivot_df[pivot_df['SHIFT DUTY'] == 'B']
    pivot_df_siftC = pivot_df[pivot_df['SHIFT DUTY'] == 'C']

    # Concatenate properly
    merge_df = pd.concat([pivot_df_siftA, pivot_df_siftB, pivot_df_siftC], axis=0)
    
    return merge_df


In [84]:
selected_day = "2025-07-25"

In [85]:
pivot = close_power_outage_duration(dataset_path, selected_day)

In [86]:
pivot

,DIVISION,SUB-DIVISION,SHIFT DUTY,2<4_Closed,4<8_Closed,<2_Closed,>8_Closed,All
0,BARGARH,BARGARH-2,A,0,1,0,0,1
1,BARGARH(W),PAIKMAL,A,1,0,0,0,1
3,BARGARH(W),SOHELA,A,0,0,1,1,2
4,BOLANGIR,BOLANGIR-1,A,1,0,0,0,1
7,BOLANGIR,LOISINGA,A,0,0,1,0,1
10,DEOGARH,DEOGARH,A,0,0,2,0,2
12,JHARSUGUDA,KUCHINDA,A,0,3,3,0,6
15,RAJGANGPUR,"SDO-1, RAJGANPUR",A,0,0,1,0,1
18,ROURKELA-SADAR,SDO-7 BONAI,A,0,1,0,0,1
20,SAMBALPUR-E,RAIRAKHOL,A,0,0,3,0,3
